# YOLO Model Format Converter
Convert between `.pt`, `.onnx`, and OpenVINO (`.xml`) formats.

**Supported conversions:**
- PT → ONNX
- PT → OpenVINO
- ONNX → OpenVINO

**Not possible:**
- ONNX → PT (PyTorch model architecture is lost during export)
- OpenVINO → PT (same reason)
- OpenVINO → ONNX (not natively supported)

**Tip:** Always keep your `.pt` file as the master — it's the only format you can convert FROM to all others.

In [ ]:
# Install dependencies (run once)
%pip install ultralytics onnxruntime openvino

In [ ]:
######################################
# CONFIG — Set your model path here  #
######################################

MODEL_PATH = r"runs\7\phase2\best.pt"  # Change this to your model
IMG_SIZE = 640

---
## 1. PT → ONNX

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_PATH)
onnx_path = model.export(format='onnx', imgsz=IMG_SIZE, simplify=True, opset=12)
print(f"Exported: {onnx_path}")

---
## 2. PT → OpenVINO

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_PATH)
openvino_path = model.export(format='openvino', imgsz=IMG_SIZE)
print(f"Exported: {openvino_path}")

---
## 3. ONNX → OpenVINO
Uses OpenVINO's model optimizer to convert an existing ONNX file.

In [ ]:
import openvino as ov
from pathlib import Path

ONNX_PATH = MODEL_PATH.replace('.pt', '.onnx')  # or set manually

core = ov.Core()
ov_model = core.read_model(ONNX_PATH)

output_dir = Path(ONNX_PATH).parent / (Path(ONNX_PATH).stem + '_openvino_model')
output_dir.mkdir(exist_ok=True)
output_xml = output_dir / (Path(ONNX_PATH).stem + '.xml')

ov.save_model(ov_model, str(output_xml))
print(f"Exported: {output_xml}")

---
## Verify Exported Models
Quick smoke test on each exported format.

In [ ]:
import numpy as np
import time

dummy_img = np.random.randint(0, 255, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

def benchmark(label, model_path, runs=5):
    m = YOLO(model_path)
    # Warmup
    m(dummy_img, imgsz=IMG_SIZE, verbose=False, device='cpu')
    
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        results = m(dummy_img, imgsz=IMG_SIZE, verbose=False, device='cpu')
        times.append((time.perf_counter() - t0) * 1000)
    
    mean_ms = sum(times) / len(times)
    print(f"{label:20s} | Mean: {mean_ms:6.1f} ms | Min: {min(times):6.1f} ms | ~{1000/mean_ms:.1f} FPS")

from ultralytics import YOLO
from pathlib import Path

pt_path = MODEL_PATH
onnx_path = MODEL_PATH.replace('.pt', '.onnx')
ov_dir = Path(MODEL_PATH).parent / (Path(MODEL_PATH).stem + '_openvino_model')
ov_path = ov_dir / (Path(MODEL_PATH).stem + '.xml')

print(f"{'Format':20s} | {'Speed':>10s} | {'Min':>10s} | {'Throughput'}")
print('-' * 65)

if Path(pt_path).exists():
    benchmark('PyTorch (.pt)', pt_path)
if Path(onnx_path).exists():
    benchmark('ONNX (.onnx)', onnx_path)
if Path(str(ov_path)).exists():
    benchmark('OpenVINO (.xml)', str(ov_path))